# 🌸 Sakuga4K — 4K Anime Video Upscaler Cloud Engine
### Free Ephemeral Nvidia T4 GPU Backend for AMV & Anime Video Editors

---

### ⚡ Step 1: Click "Run all" (Runtime ➔ Run all)
This single step will:
1. Verify your free Nvidia T4 GPU (16 GB VRAM).
2. Mount your Google Drive for permanent 4K auto-sync.
3. Start the FastAPI neural backend in half-precision (FP16).
4. Generate your **Cloudflare Quick Tunnel URL** and **Scannable QR Code**.

Copy the printed link and paste it into your Sakuga4K web studio!

In [ ]:
# @title 🚀 1-Click Master Engine (Hit Play Button)
!nvidia-smi

# 1. Mount Google Drive for automatic permanent output sync
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# 2. Clone / Pull Repository & Install Dependencies
import os, sys
if not os.path.exists('/content/Sakuga4k'):
    !git clone https://github.com/Anyone-design/Sakuga4k.git /content/Sakuga4k
os.chdir('/content/Sakuga4k')
!git fetch origin main && git reset --hard origin/main
!pip install -q -r backend/requirements.txt qrcode[pil]

# 2b. Cache Neural Model Weights for GPU Tensor Cores
!mkdir -p /content/models
!wget -q -nc -O /content/models/realesr-animevideov3.pth https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-animevideov3.pth || true
!wget -q -nc -O /content/models/RealESRGAN_x4plus_anime_6B.pth https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth || true

# 3. Install Cloudflare Quick Tunnel
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
!rm -f cloudflared-linux-amd64.deb

# 4. Launch FastAPI Uvicorn with Diagnostics & Self-Healing
import subprocess, time, re, sys, urllib.request, qrcode

# Terminate old instances if restarting cell
!pkill -f uvicorn > /dev/null 2>&1 || true
!pkill -f cloudflared > /dev/null 2>&1 || true

backend_dir = '/content/Sakuga4k/backend'
if backend_dir not in sys.path:
    sys.path.insert(0, backend_dir)
env = os.environ.copy()
env['PYTHONPATH'] = f"{backend_dir}:{env.get('PYTHONPATH', '')}"

log_file = open('/content/uvicorn.log', 'w')
server_process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app", "--app-dir", backend_dir, "--host", "0.0.0.0", "--port", "8000"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=env
)

# Pre-flight check: Verify local FastAPI responds before opening tunnel
server_ready = False
for _ in range(15):
    time.sleep(1)
    if server_process.poll() is not None:
        print("❌ FastAPI server exited unexpectedly! Log:")
        with open('/content/uvicorn.log') as f:
            print(f.read())
        break
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/api/health", timeout=2) as resp:
            if resp.status == 200:
                server_ready = True
                break
    except Exception:
        pass

if not server_ready:
    print("❌ FastAPI health check timed out. Output log:")
    with open('/content/uvicorn.log') as f:
        print(f.read())
    raise RuntimeError("FastAPI backend failed to start")

print("⚡ Local FastAPI Engine is healthy and responding on port 8000!")

# 5. Establish Tunnel & Print QR / Link
tunnel_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True
)

print("\n=======================================================")
print("🌸 Sakuga4K Cloud Tunnel Starting...")
print("=======================================================\n")

tunnel_url = None
for line in tunnel_process.stdout:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    print("\n✅ SUCCESS! Paste this URL into your Sakuga4K Studio Web App:")
    print(f"\n👉  {tunnel_url}  👈\n")
    
    # Print ASCII QR code for instant mobile phone pairing
    qr = qrcode.QRCode()
    qr.add_data(tunnel_url)
    qr.print_ascii(invert=True)
    print("\n⚡ Keep this Colab tab running while upscaling clips!")
    print("🔥 Live GPU Neural Inference Monitor (Real-time telemetry):")
    print("-------------------------------------------------------\n")
    import threading
    def stream_server_logs():
        with open('/content/uvicorn.log', 'r') as f:
            f.seek(0, os.SEEK_END)
            while True:
                line = f.readline()
                if line:
                    text = line.strip()
                    if "[Sakuga4K" in text or "POST /api/upload" in text:
                        print(f">>> {text}", flush=True)
                else:
                    time.sleep(0.1)
    log_thread = threading.Thread(target=stream_server_logs, daemon=True)
    log_thread.start()
    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        print("\n🛑 Master Engine stopped by user.")
else:
    print("❌ Tunnel failed to start. Check /content/uvicorn.log")


In [ ]:
# @title ⚡ Live GPU Tensor Diagnostic & VRAM Benchmark
# Run this cell anytime to verify GPU CUDA acceleration and VRAM allocation directly!
import torch
import sys
backend_path = '/content/Sakuga4k/backend'
if backend_path not in sys.path:
    sys.path.insert(0, backend_path)
from models import model_manager

print('🎮 GPU Hardware:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No CUDA detected')
print('📦 Testing Real-ESRGAN / Real-CUGAN weights on Tensor Cores...')
model = model_manager.load_model('animevideov3')
dummy_frame = torch.rand(1, 3, 270, 480)
upscaled = model_manager.upscale_tensor(dummy_frame, preset='animevideov3')
vram_mb = torch.cuda.memory_allocated(0) / (1024 * 1024) if torch.cuda.is_available() else 0
print(f'✅ CUDA SUCCESS! Input: [1, 3, 270, 480] -> Output: {list(upscaled.shape)}')
print(f'⚡ Active GPU VRAM Allocation: {vram_mb:.1f} MB')
!nvidia-smi --query-gpu=name,memory.total,memory.used,utilization.gpu --format=csv
